# Capstone Project: Claude Certified Architect – Foundations

**Scenario**: You are the architect for a customer support resolution agent (drawn from the exam's
Scenario 1) built with the Claude Agent SDK. It handles high-ambiguity requests (returns, billing
disputes, account issues) via custom MCP tools (`get_customer`, `lookup_order`, `process_refund`,
`escalate_to_human`), configured and developed through Claude Code, with a target of 80%+
first-contact resolution.

**Domains exercised** (one milestone per domain, in order):
1. Agentic Architecture & Orchestration
2. Tool Design & MCP Integration
3. Claude Code Configuration & Workflows
4. Prompt Engineering & Structured Output
5. Context Management & Reliability

**Prerequisites**: complete all 5 domain notebooks first.

**Estimated time**: 5-8 hours (deep depth).

In [ ]:
# Setup: these notebooks demonstrate Claude Agent SDK / Claude API / MCP concepts using
# plain Python simulations of the relevant objects and control flow, so every cell is runnable
# without needing a live Claude Agent SDK installation or API key. Where call_claude() is used,
# it falls back to a simulated response if ANTHROPIC_API_KEY is not set.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception:
        return f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY to call the real API] Prompt: {prompt[:120]}..."

print("call_claude() ready. Concepts below use plain Python to simulate Agent SDK / MCP / Claude Code behavior.")

## Milestone 1: Agentic Architecture & Orchestration

Implement the agentic loop and a hook enforcing identity verification before refunds.

In [ ]:
verified_customers = {"C1": "Alex Rivera"}

def get_customer(name):
    for cid, cname in verified_customers.items():
        if cname.lower() == name.lower():
            return cid
    return None

def pre_tool_call_hook(tool_name, tool_input, verified_customer_id):
    if tool_name == "process_refund" and not verified_customer_id:
        return False
    if tool_name == "process_refund" and tool_input.get("amount", 0) > 500:
        return False
    return True

# TODO: implement handle_support_request(customer_name, amount) using get_customer + pre_tool_call_hook,
# returning either a refund confirmation string or an escalation string
def handle_support_request(customer_name: str, amount: float) -> str:
    pass

print(handle_support_request("Alex Rivera", 150))
print(handle_support_request("Unknown", 150))

In [ ]:
print("Milestone 1 complete.")  # Verification: re-run the cell above with no errors

## Milestone 2: Tool Design & MCP Integration

Design clear tool descriptions and structured errors for the 4 MCP tools.

In [ ]:
tool_descriptions = {
    "get_customer": "get_customer(name_or_id: str) -> Customer. Use to verify identity before any account-changing operation.",
    "lookup_order": "lookup_order(order_id: str) -> Order. Use ONLY after get_customer has returned a verified ID.",
    "process_refund": "process_refund(customer_id: str, amount: float) -> RefundResult. Requires a verified customer_id.",
    "escalate_to_human": "escalate_to_human(summary: str) -> None. Use when policy is ambiguous or the customer requests a human.",
}
# TODO: write a structured error response (isError/errorCategory/isRetryable) for a lookup_order
# call made with an invalid order_id format
def lookup_order_error(order_id: str):
    pass

print(lookup_order_error("bad-id"))

In [ ]:
print("Milestone 2 complete.")  # Verification: re-run the cell above with no errors

## Milestone 3: Claude Code Configuration & Workflows

Set up CLAUDE.md and a skill for the team building this agent.

In [ ]:
claude_md_content = "# support-agent conventions\nAll tool calls must be logged. Refunds >$500 always escalate."
skill_frontmatter = """---
name: refund-policy-check
context: fork
allowed-tools: [Read]
---
"""
print(claude_md_content)
print(skill_frontmatter)
# TODO: decide plan mode vs direct execution for adding a 5th MCP tool (loyalty_credit) to this agent
decision = "..."  # TODO
print(decision)

In [ ]:
print("Milestone 3 complete.")  # Verification: re-run the cell above with no errors

## Milestone 4: Prompt Engineering & Structured Output

Design a structured handoff schema for escalation to a human agent.

In [ ]:
# TODO: design a JSON schema (as a dict) for the escalate_to_human handoff payload:
# customer_id, root_cause, attempted_actions (list), recommended_action
handoff_schema = {}  # TODO
print(handoff_schema)

In [ ]:
print("Milestone 4 complete.")  # Verification: re-run the cell above with no errors

## Milestone 5: Context Management & Reliability

Extract persistent case facts and apply the escalation decision function across a multi-issue request.

In [ ]:
multi_issue_notes = ["Order #771 damaged on arrival", "Customer wants a $600 refund", "Customer has not asked for a human yet"]

def extract_case_facts(notes):
    facts = {}
    for n in notes:
        if "order" in n.lower():
            facts["order_id"] = n.split("#")[-1].split()[0]
        if "$" in n:
            facts["amount"] = n.split("$")[-1].split()[0]
    return facts

facts = extract_case_facts(multi_issue_notes)
print(facts)

def should_escalate(customer_demands_human, policy_covers_case, can_make_progress):
    return customer_demands_human or not policy_covers_case or not can_make_progress

# TODO: given facts["amount"] > 500 falls outside standard refund policy, decide whether to escalate
print(should_escalate(False, int(facts.get("amount", 0)) <= 500, True))

In [ ]:
print("Milestone 5 complete.")  # Verification: re-run the cell above with no errors

## Integration

The 5 milestones connect into one support agent: an agentic loop enforcing identity/amount rules
via hooks (M1), backed by well-described, error-safe MCP tools (M2), developed and configured
through Claude Code with team conventions (M3), with a structured escalation handoff schema (M4),
and context/fact extraction plus escalation logic to handle ambiguous multi-issue requests (M5).

In [ ]:
# TODO: implement run_support_agent(customer_name, notes) that combines all 5 milestones:
# extract facts -> verify identity -> check refund policy/escalation -> return a resolution or handoff payload
def run_support_agent(customer_name: str, notes: list[str]) -> dict:
    pass

print(run_support_agent("Alex Rivera", multi_issue_notes))

## Stretch Goals (deep depth)

1. Add a second subagent (e.g., a policy-lookup agent) coordinated via the Task tool pattern from Domain 1.
2. Add a `PostToolUse` hook that normalizes heterogeneous order-status formats across 2 fake backend systems.
3. Add stratified confidence-based routing (Domain 5) for automatically-resolved vs. escalated cases.
4. Wire the agent into a simulated CI pipeline that runs a Claude Code review on any change to the hook logic.
5. Implement a Message Batches API-based nightly job that re-analyzes all escalated cases from the day for policy-gap patterns.

## Final Verification Checklist

- [ ] Agentic loop with `stop_reason`-driven control flow implemented
- [ ] Identity-verification hook blocks unverified refunds
- [ ] All 4 MCP tools have clear, non-overlapping descriptions
- [ ] Structured error response implemented for at least one tool
- [ ] CLAUDE.md and at least one skill configured for the team
- [ ] Structured escalation handoff schema designed
- [ ] Case-fact extraction and escalation decision logic implemented
- [ ] `run_support_agent` implemented and runs end-to-end

---
**Back**: [README](./README.md)